# Demo experiment, May 4, Francesco Braicovich

**Trait:** politeness  
**Model:** `google/gemma-2-2b` (base, completion)  
**Layer:** 13, central (there are 26 layers in total)  
**Data:** accepted paraphrases from the smoke-test run (`output_smoke/politeness/accepted.jsonl`)

Unlike Bettineschi's notebook (which used hand-written single-sentence prompts), this notebook
loads the LLM-generated ordinal paraphrases produced by the data generation pipeline and checks
whether the low → mid → high politeness ladder leaves a linear trace in activation space.

In [ ]:
import json
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path("..").resolve()))  # src/notebooks/ -> src/

from data import Sample
from representations import load_model, extract_activations, extract_activations_multilayer
from analysis import compute_difference_vectors, similarity_matrix, plot_similarity_matrix

In [ ]:
MODEL   = "google/gemma-2-2b"
LAYER   = 13
TRAIT   = "politeness"
ACCEPTED_JSONL = Path("../../output_smoke/politeness/accepted.jsonl")

In [ ]:
records = []
with ACCEPTED_JSONL.open() as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

LEVELS = sorted({r["level"] for r in records})
print(f"Loaded {len(records)} accepted paraphrases")
print(f"Levels found: {LEVELS}")
for lvl in LEVELS:
    texts = [r["text"] for r in records if r["level"] == lvl]
    print(f"  {lvl}: {len(texts)} items — e.g. '{texts[0][:80]}'")

Loaded 3 accepted paraphrases
Levels found: ['low']
  low: 3 items — e.g. 'Send the latest budget spreadsheet for project X.'


In [ ]:
samples = [Sample(prompt=r["text"], trait=r["trait"], intensity=r["level"]) for r in records]
TRAITS_CFG = [{"name": TRAIT, "intensities": LEVELS}]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

device: mps


In [ ]:
# slow — run once
model, tokenizer = load_model(MODEL, device)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b.
403 Client Error. (Request ID: Root=1-69fa0cfd-0d2823ea4f8d657733616cff;12f5c0eb-3cba-4d4d-9906-2a2801bcad31)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b/resolve/main/config.json.
Access to model google/gemma-2-2b is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2-2b to ask for access.

In [ ]:
activations = extract_activations(samples, model, tokenizer, LAYER, device)

for (trait, intensity), vec in activations.items():
    print(f"  ({trait}, {intensity}): shape={vec.shape}, norm={vec.norm():.2f}")

In [ ]:
diffs = compute_difference_vectors(activations, TRAITS_CFG)

for (trait, lo, hi), vec in diffs.items():
    print(f"  {trait}: {lo} -> {hi}  norm={vec.norm():.2f}")

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

labels, matrix = similarity_matrix(diffs)

print("Cosine similarity matrix:")
header = f"{'':>25}" + "".join(f"{l:>27}" for l in labels)
print(header)
for i, li in enumerate(labels):
    row = f"{li:>25}" + "".join(f"{matrix[i, j]:>27.4f}" for j in range(len(labels)))
    print(row)

if len(labels) >= 2:
    print(f"\nKey number — cosine similarity {labels[0]} vs {labels[1]}: {matrix[0, 1]:.4f}")
    print("  +1 = same direction (linear), -1 = opposite, ~0 = unrelated")

In [ ]:
n = len(labels)
cell = 2.2
fig, ax = plt.subplots(figsize=(max(6, n * cell), max(6, n * cell)))
im = ax.imshow(matrix, vmin=-1, vmax=1, cmap="RdYlGn")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(labels, fontsize=11)
for i, j in itertools.product(range(n), range(n)):
    color = "white" if abs(matrix[i, j]) > 0.65 else "black"
    ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center",
            fontsize=13, fontweight="bold", color=color)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("cosine similarity", fontsize=10)
ax.set_title(f"Cosine similarity of activation difference vectors — {TRAIT}", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

In [ ]:
sweep_layers = list(range(8, 23))

# One forward pass per batch, all layers captured at once.
acts_all = extract_activations_multilayer(samples, model, tokenizer, sweep_layers, device)

sims = []
for layer in sweep_layers:
    acts_layer = {(t, i): v for (t, i, l), v in acts_all.items() if l == layer}
    d = compute_difference_vectors(acts_layer, TRAITS_CFG)
    _, m = similarity_matrix(d)
    sims.append(m[0, 1])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(sweep_layers, sims, marker="o", linewidth=1.5)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Layer")
ax.set_ylabel("Cosine similarity")
ax.set_title(f"{TRAIT} direction alignment by layer ({labels[0]} vs {labels[1]})")
plt.tight_layout()
plt.show()